# 技能0 · Day 1 上机：Python 编程基础 + 营销数据处理

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pandas** 加载营销数据（产品/客户/订单）为 DataFrame
2. 用 `dtypes`/`describe()`/`info()` 完成数据类型检查与探索性分析
3. 用 Python 控制流和函数实现 RFM 客户分类逻辑，用 `apply` 向量化执行
4. 用 Python 类设计营销 Product 和 Customer 对象
5. 用 pandas 读写 CSV/JSON 文件，计算核心营销指标（ROI/转化率/客单价/复购率）

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：pandas（pandas-dev/pandas，43k+ star）+ numpy。
营销映射：8个产品 × 15个客户 × 30笔订单，用 Python 完成从数据加载到营销指标计算的完整闭环。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> 需要 pandas 和 numpy。通常已随 conda/venv 安装。
> pandas 2.x 推荐：基于 Apache Arrow 的新后端可提升内存效率 30-50%。

In [ ]:
# !pip install pandas numpy -q


## 1. 数据集背景与营销映射

**处理对象**：真实营销场景的产品/客户/订单数据（8个产品 × 15个客户 × 30笔订单）。

| 数据表 | 记录数 | 核心字段 | 营销用途 |
|--------|--------|---------|---------|
| 产品表 products | 8 | product_id, product_name, category, unit_price, unit_cost | 产品管理、利润分析 |
| 客户表 customers | 15 | customer_id, age, gender, registration_date, channel | 客户画像、获客渠道 |
| 订单表 orders | 30 | order_id, customer_id, product_id, quantity, order_date, discount | 消费行为、RFM分析 |

**产品类别**：skincare（护肤）、electronics（电子）、fitness（健身）

**营销映射**：在真实项目中，这些数据来自电商平台的 CRM/ERP 系统。Python + pandas 的目标是把原始数据转化为可计算的营销指标，为 AI 营销方案提供数据支撑。

**理论连接**：Python 五大语法要素在营销数据处理中的映射——变量与数据类型（销售额 float、客户ID str）、控制流（筛选高价值客户）、函数（RFM 分类逻辑封装）、数据结构（dict 处理 JSON API）、面向对象（Product/Customer 业务对象设计）。

In [ ]:
import pandas as pd
import numpy as np
import json
import csv
from datetime import datetime

# ===== 营销数据：产品/客户/订单 =====

# 产品数据（8个产品，3个类别）
products_data = {
    'product_id': ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008'],
    'product_name': ['烟酰胺精华液', '保湿面霜', '防晒霜SPF50', '跑步手表',
                     '蓝牙耳机', '智能体脂秤', '瑜伽垫', '阻力带套装'],
    'category': ['skincare', 'skincare', 'skincare',
                 'electronics', 'electronics', 'electronics',
                 'fitness', 'fitness'],
    'unit_price': [299, 159, 129, 899, 499, 169, 89, 69],
    'unit_cost': [120, 65, 55, 450, 220, 75, 35, 28]
}

# 客户数据（15个客户）
customers_data = {
    'customer_id': ['C0001', 'C0002', 'C0003', 'C0004', 'C0005',
                    'C0006', 'C0007', 'C0008', 'C0009', 'C0010',
                    'C0011', 'C0012', 'C0013', 'C0014', 'C0015'],
    'age': [28, 35, 42, 31, 26, 38, 45, 29, 33, 41, 24, 37, 30, 44, 27],
    'gender': ['F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M',
               'F', 'M', 'F', 'M', 'F'],
    'registration_date': ['2024-01-15', '2024-02-20', '2024-01-08', '2024-03-10',
                          '2024-02-14', '2024-01-22', '2024-03-05', '2024-02-28',
                          '2024-01-30', '2024-02-10', '2024-03-15', '2024-01-18',
                          '2024-02-25', '2024-03-01', '2024-02-05'],
    'channel': ['wechat', 'douyin', 'wechat', 'taobao', 'douyin',
                'wechat', 'taobao', 'douyin', 'wechat', 'taobao',
                'douyin', 'wechat', 'taobao', 'wechat', 'douyin']
}

# 订单数据（30笔订单）
orders_data = {
    'order_id': ['ORD001', 'ORD002', 'ORD003', 'ORD004', 'ORD005',
                 'ORD006', 'ORD007', 'ORD008', 'ORD009', 'ORD010',
                 'ORD011', 'ORD012', 'ORD013', 'ORD014', 'ORD015',
                 'ORD016', 'ORD017', 'ORD018', 'ORD019', 'ORD020',
                 'ORD021', 'ORD022', 'ORD023', 'ORD024', 'ORD025',
                 'ORD026', 'ORD027', 'ORD028', 'ORD029', 'ORD030'],
    'customer_id': ['C0001', 'C0002', 'C0003', 'C0001', 'C0004',
                    'C0005', 'C0002', 'C0006', 'C0003', 'C0007',
                    'C0008', 'C0001', 'C0009', 'C0005', 'C0010',
                    'C0011', 'C0006', 'C0012', 'C0003', 'C0013',
                    'C0007', 'C0014', 'C0009', 'C0002', 'C0015',
                    'C0004', 'C0011', 'C0012', 'C0008', 'C0010'],
    'product_id': ['P001', 'P004', 'P002', 'P007', 'P005',
                   'P003', 'P008', 'P004', 'P006', 'P001',
                   'P005', 'P004', 'P002', 'P007', 'P004',
                   'P003', 'P008', 'P001', 'P005', 'P002',
                   'P006', 'P004', 'P007', 'P001', 'P003',
                   'P008', 'P005', 'P002', 'P006', 'P001'],
    'quantity': [2, 1, 3, 1, 1, 2, 2, 1, 1, 1,
                 2, 1, 2, 3, 1, 1, 2, 2, 1, 1,
                 2, 1, 2, 1, 2, 1, 1, 3, 1, 2],
    'order_date': ['2024-03-10', '2024-03-15', '2024-02-20', '2024-04-05', '2024-03-22',
                   '2024-04-10', '2024-05-01', '2024-03-18', '2024-04-15', '2024-03-25',
                   '2024-04-20', '2024-05-15', '2024-03-30', '2024-05-05', '2024-04-08',
                   '2024-04-25', '2024-05-10', '2024-03-12', '2024-05-20', '2024-04-02',
                   '2024-05-08', '2024-03-28', '2024-05-12', '2024-06-01', '2024-04-18',
                   '2024-05-25', '2024-05-28', '2024-04-12', '2024-06-05', '2024-05-30'],
    'channel': ['wechat', 'douyin', 'wechat', 'wechat', 'taobao',
                'douyin', 'douyin', 'wechat', 'wechat', 'taobao',
                'douyin', 'wechat', 'wechat', 'douyin', 'taobao',
                'douyin', 'wechat', 'wechat', 'wechat', 'taobao',
                'taobao', 'wechat', 'wechat', 'douyin', 'douyin',
                'taobao', 'douyin', 'wechat', 'douyin', 'taobao'],
    'discount': [0.10, 0.05, 0.00, 0.15, 0.10,
                 0.05, 0.00, 0.10, 0.05, 0.00,
                 0.10, 0.05, 0.00, 0.15, 0.10,
                 0.05, 0.00, 0.10, 0.05, 0.00,
                 0.10, 0.05, 0.00, 0.10, 0.05,
                 0.00, 0.10, 0.05, 0.00, 0.10]
}

print("数据定义完成")
print(f"产品数: {len(products_data['product_id'])}")
print(f"客户数: {len(customers_data['customer_id'])}")
print(f"订单数: {len(orders_data['order_id'])}")

## 1：用 pandas 加载营销数据

**任务**：将产品/客户/订单数据（已定义为 Python 字典）加载为 pandas DataFrame。

**提示**：
- `pd.DataFrame(dict)` 从字典创建 DataFrame
- `df.shape` 返回 (行数, 列数)
- `df.head(n)` 返回前 n 行

**理论连接**：DataFrame 是 pandas 的核心数据结构，可以理解为 Excel 表格的程序化版本，但能处理百万行级别数据。选择 pandas 不是偏好问题，是生态问题——后续的 NumPy、scikit-learn、matplotlib 全部与 pandas 无缝衔接。

In [ ]:
# 1. 用 pandas 将产品/客户/订单数据加载为 DataFrame
products_df = pd.DataFrame(products_data)
customers_df = pd.DataFrame(customers_data)
orders_df = pd.DataFrame(orders_data)

print(f"产品表形状: {products_df.shape}")
print(f"客户表形状: {customers_df.shape}")
print(f"订单表形状: {orders_df.shape}")
print("\n--- 产品表前5行 ---")
print(products_df.head())
print("\n--- 客户表前5行 ---")
print(customers_df.head())
print("\n--- 订单表前5行 ---")
print(orders_df.head())

## 2. Python 基础语法回顾

### 五大语法要素与营销映射

| 语法要素 | 关键字 | 营销应用 |
|---------|--------|---------|
| 变量与数据类型 | int/float/str/bool | 销售额 float、客户ID str（前导零）、是否复购 bool |
| 控制流 | if-elif-else/for | 遍历客户列表分群、if 筛选高价值客户 |
| 函数 | def/return/apply | 封装 RFM 分类逻辑、`df.apply()` 向量化 |
| 数据结构 | list/dict/tuple/set | dict 处理 JSON API 返回值 |
| 面向对象 | class/属性/方法 | 设计 Product/Customer 类 |

### pandas DataFrame 核心 API

| 操作 | 方法 | 用途 |
|------|------|------|
| 类型检查 | `df.dtypes` | 检查数据类型是否正确 |
| 探索统计 | `df.describe()` | 均值/标准差/分位数 |
| 信息概览 | `df.info()` | 行数/列数/类型/内存 |
| 筛选 | `df[df['col'] > val]` | 条件筛选 |
| 分组聚合 | `df.groupby('col').agg(...)` | 按类别统计 |

**关键注意**：客户ID（如 C0001）必须存储为 str 而非 int，否则前导零会丢失。这是数据治理的基本要求。

## 2：数据类型与探索性分析

**任务**：检查三个 DataFrame 的数据类型、描述性统计和信息概览。

**提示**：
- `df.dtypes` 返回各列数据类型
- `df.describe()` 返回数值列的统计信息（均值/标准差/最小值/最大值/分位数）
- `df.info()` 打印行数/列数/类型/内存使用

**数据治理视角**：注意检查 customer_id 是否为 str（不能是 int，否则前导零丢失）、order_date 是否需要转换为 datetime 类型、discount 的范围是否合理（0-1之间）。

In [ ]:
# 2. 数据类型与探索性分析
dtypes_products = products_df.dtypes
dtypes_customers = customers_df.dtypes
desc_products = products_df.describe()
desc_orders = orders_df.describe()

print("=== 产品表数据类型 ===")
print(dtypes_products)
print("\n=== 客户表数据类型 ===")
print(dtypes_customers)
print("\n=== 产品表描述性统计 ===")
print(desc_products)
print("\n=== 订单表描述性统计 ===")
print(desc_orders)
print("\n=== 订单表信息 ===")
orders_df.info()

## 3：控制流与函数 -- RFM 客户分类

**任务**：用控制流和函数实现 RFM 客户分类逻辑。

**RFM 理论**：
- **R（Recency）**：最近一次购买距今天数 -- 越小越好
- **F（Frequency）**：购买次数 -- 越大越好
- **M（Monetary）**：消费总金额 -- 越大越好

**分类规则**（简化版）：
- 高价值客户：R ≤ 30天 且 F ≥ 3次 且 M ≥ ¥1000
- 中等价值客户：R ≤ 60天 且 F ≥ 2次
- 低价值客户：R ≤ 90天
- 流失风险客户：其他

**提示**：
1. 用 `pd.merge(orders_df, products_df, on='product_id')` 合并订单与产品
2. 计算 `revenue = quantity * unit_price * (1 - discount)`
3. 用 `groupby('customer_id').agg(...)` 计算 R/F/M
4. 定义 `classify_customer(row)` 函数，用 `if-elif-else` 实现分类
5. 用 `rfm.apply(classify_customer, axis=1)` 向量化应用

**为什么用 apply 而非 for 循环**：`apply` 底层用 C 实现，比原生 Python for 循环快 10-100 倍。在百万行数据上，这个差异从秒级变成分钟级。

In [ ]:
# 3. 控制流与函数 -- RFM 客户分类
REFERENCE_DATE = '2024-06-30'

# 合并订单与产品数据
merged = pd.merge(orders_df, products_df, on='product_id')

# 计算每笔订单的收入
merged['revenue'] = merged['quantity'] * merged['unit_price'] * (1 - merged['discount'])

# 将 order_date 转为 datetime
merged['order_date'] = pd.to_datetime(merged['order_date'])

# 计算 RFM 指标
rfm = merged.groupby('customer_id').agg(
    recency=('order_date', lambda x: (pd.to_datetime(REFERENCE_DATE) - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('revenue', 'sum')
).reset_index()

# 定义客户分类函数
def classify_customer(row):
    r, f, m = row['recency'], row['frequency'], row['monetary']
    if r <= 30 and f >= 3 and m >= 1000:
        return '高价值客户'
    elif r <= 60 and f >= 2:
        return '中等价值客户'
    elif r <= 90:
        return '低价值客户'
    else:
        return '流失风险客户'

# 应用分类
rfm['segment'] = rfm.apply(classify_customer, axis=1)

print("=== RFM 客户分群结果 ===")
print(rfm[['customer_id', 'recency', 'frequency', 'monetary', 'segment']].to_string(index=False))
print("\n=== 各层级客户数 ===")
print(rfm['segment'].value_counts())

## 4：类与面向对象 -- 设计营销业务对象

**任务**：设计 Product 和 Customer 类，包含属性和方法。

**为什么学面向对象**：后续技能中的 LangChain（Agent 编排）、DoWhy（因果推断）、PyTorch（深度学习）等库的 API 全部是面向对象设计的。理解 class/`__init__`/self/方法是使用这些库的前提。

**Product 类设计**：
- 属性：product_id, product_name, category, unit_price, unit_cost
- 方法：`margin()` 返回单位利润（price - cost）、`margin_rate()` 返回利润率
- `__repr__()` 返回字符串表示

**Customer 类设计**：
- 属性：customer_id, age, gender, registration_date, channel, orders（列表）
- 方法：`add_order(revenue)` 添加订单、`total_spending()` 返回总消费、`is_repeat_buyer()` 返回是否复购
- `__repr__()` 返回字符串表示

In [ ]:
# 4. 类与面向对象 -- 设计营销 Product 和 Customer 类
class Product:
    def __init__(self, product_id, product_name, category, unit_price, unit_cost):
        self.product_id = product_id
        self.product_name = product_name
        self.category = category
        self.unit_price = unit_price
        self.unit_cost = unit_cost

    def margin(self):
        return self.unit_price - self.unit_cost

    def margin_rate(self):
        return self.margin() / self.unit_price

    def __repr__(self):
        return f"Product({self.product_id}, {self.product_name}, ¥{self.unit_price})"

class Customer:
    def __init__(self, customer_id, age, gender, registration_date, channel):
        self.customer_id = customer_id
        self.age = age
        self.gender = gender
        self.registration_date = registration_date
        self.channel = channel
        self.orders = []

    def add_order(self, revenue):
        self.orders.append(revenue)

    def total_spending(self):
        return sum(self.orders)

    def is_repeat_buyer(self):
        return len(self.orders) > 1

    def __repr__(self):
        return f"Customer({self.customer_id}, age={self.age}, {self.gender})"

# 创建产品对象
p1 = Product('P001', '烟酰胺精华液', 'skincare', 299, 120)
p4 = Product('P004', '跑步手表', 'electronics', 899, 450)

# 创建客户对象并添加订单
c1 = Customer('C0001', 28, 'F', '2024-01-15', 'wechat')
c1.add_order(2 * 299 * 0.9)   # ORD001: 2 * P001 * (1-0.10)
c1.add_order(1 * 89 * 0.85)   # ORD004: 1 * P007 * (1-0.15)
c1.add_order(1 * 899 * 0.95)  # ORD012: 1 * P004 * (1-0.05)

print(f"产品1: {p1}")
print(f"  利润: ¥{p1.margin()}, 利润率: {p1.margin_rate():.2%}")
print(f"产品4: {p4}")
print(f"  利润: ¥{p4.margin()}, 利润率: {p4.margin_rate():.2%}")
print(f"\n客户1: {c1}")
print(f"  总消费: ¥{c1.total_spending():.2f}")
print(f"  是否复购: {c1.is_repeat_buyer()}")

## 5：文件 IO -- 读写营销 CSV/JSON

**任务**：将产品表和 RFM 分析结果写入 CSV 和 JSON 文件，再读回验证。

**CSV vs JSON**：
- **CSV**：表格型数据标准格式，Excel/数据库导出常用。优点：简单通用；缺点：不支持嵌套
- **JSON**：树形结构数据标准格式，API 返回值常用。优点：支持嵌套、自描述；缺点：比CSV占更多空间

**提示**：
- `df.to_csv('file.csv', index=False, encoding='utf-8-sig')` -- `utf-8-sig` 解决中文 Excel 乱码
- `df.to_json('file.json', orient='records', force_ascii=False)` -- `force_ascii=False` 保留中文
- `pd.read_csv()` / `pd.read_json()` 读回验证

**营销场景**：在真实项目中，分析结果需要导出给市场部使用。CSV 供 Excel 打开，JSON 供 API 调用。

In [ ]:
# 5. 文件 IO -- 读写营销 CSV/JSON
import os
os.makedirs('output', exist_ok=True)

# 写入 CSV
products_csv = 'output/products.csv'
products_df.to_csv(products_csv, index=False, encoding='utf-8-sig')

# 写入 JSON
rfm_json = 'output/rfm.json'
rfm.to_json(rfm_json, orient='records', force_ascii=False)

# 读回验证
products_loaded = pd.read_csv(products_csv)
rfm_loaded = pd.read_json(rfm_json, orient='records')

print(f"CSV 已写入: {products_csv}")
print(f"JSON 已写入: {rfm_json}")
print(f"\n读回的产品表形状: {products_loaded.shape}")
print(f"读回的 RFM 表形状: {rfm_loaded.shape}")
print(f"\n读回的产品表前3行:")
print(products_loaded.head(3))

## 6：营销指标计算 -- ROI/转化率/客单价/复购率

**任务**：计算四个核心营销指标。

**指标定义**：
- **ROI（投资回报率）**= (总收入 - 总成本) / 总成本 -- 衡量营销活动盈利能力
- **转化率**= 下单客户数 / 总客户数 -- 衡量获客效率
- **客单价 AOV（Average Order Value）**= 总收入 / 订单数 -- 衡量单次消费水平
- **复购率**= 复购客户数 / 下单客户数 -- 衡量客户忠诚度

**提示**：
1. 合并 orders_df 和 products_df 获取单价和成本
2. 计算 `revenue = quantity * unit_price * (1 - discount)` 和 `cost = quantity * unit_cost`
3. 用 `merged['revenue'].sum()` 计算总收入，`merged['cost'].sum()` 计算总成本
4. 用 `merged['customer_id'].nunique()` 计算下单客户数
5. 用 `merged.groupby('customer_id')['order_id'].count()` 计算每客户订单数，`> 1` 即复购

**营销桥接**：这些指标是营销决策的基础。ROI 高的产品应加大投放，转化率低的渠道需要优化，复购率低说明需要客户留存策略。在后续技能3（因果推断）中，你将学习如何用 DML 评估这些策略的真实因果效果。

In [ ]:
# 6. 营销指标计算 -- ROI/转化率/客单价/复购率
# 合并订单与产品数据
merged = pd.merge(orders_df, products_df, on='product_id')
merged['revenue'] = merged['quantity'] * merged['unit_price'] * (1 - merged['discount'])
merged['cost'] = merged['quantity'] * merged['unit_cost']

# ROI
total_revenue = merged['revenue'].sum()
total_cost = merged['cost'].sum()
roi = (total_revenue - total_cost) / total_cost

# 转化率
purchasing_customers = merged['customer_id'].nunique()
total_customers = len(customers_df)
conversion_rate = purchasing_customers / total_customers

# 客单价 AOV
aov = total_revenue / len(merged)

# 复购率
customer_order_counts = merged.groupby('customer_id')['order_id'].count()
repeat_customers = (customer_order_counts > 1).sum()
repurchase_rate = repeat_customers / purchasing_customers

print("=" * 50)
print("营销核心指标")
print("=" * 50)
print(f"总收入: ¥{total_revenue:,.2f}")
print(f"总成本: ¥{total_cost:,.2f}")
print(f"ROI: {roi:.2%}")
print(f"转化率: {conversion_rate:.2%} ({purchasing_customers}/{total_customers})")
print(f"客单价 AOV: ¥{aov:,.2f}")
print(f"复购率: {repurchase_rate:.2%} ({repeat_customers}/{purchasing_customers})")

## 3. 反思与前沿

### 反思问题
1. 你的营销数据在探索性分析中发现了什么数据质量问题？（如类型错误、缺失值、异常值）
2. RFM 分层后各层级客户占比如何？高价值客户有多少？
3. ROI 最高的产品类别是哪个？利润率最高的是哪个？
4. 复购率是多少？哪些客户是复购客户？如何提升复购率？

### 2026 前沿：pandas 2.x + Apache Arrow + Polars

**pandas 2.x 与 Apache Arrow**：pandas 2.0（2023年发布）引入了基于 Apache Arrow 的后端（`dtype_backend="pyarrow"`），这是 pandas 自 2008 年诞生以来最大的架构升级。Arrow 的列式存储和字典编码比 NumPy 数组节省 30-50% 内存，零拷贝数据交换让 pandas 与 Polars/DuckDB 之间无需序列化即可传递数据。

**Polars 高性能替代**：Polars（Rust 编写，28k+ star）API 设计参考 pandas 但性能提升 5-30 倍。LazyFrame 支持查询优化器，多线程自动并行化分组聚合。当营销数据超过 1GB 时，Polars 的优势显著。但 pandas 仍是学习首选，因为生态更完整。

**可复现研究与数据治理**：用 `random_state=42` 固定随机种子、用 `requirements.txt` 锁定版本、用虚拟环境隔离--确保任何人都能复现你的分析结果。pandas 2.x 的严格类型系统（`int64` vs `Int64` 可空整数）帮助在数据加载阶段就发现类型问题，是数据治理的基础。

> 参考阅读见 [reading.md](./reading.md) 的 pandas 2.x / Apache Arrow / Polars 条目。